# 05 — XGBoost valuation model (honest split-conformal)

Trains the price model on 672 real Dubizzle UAE listings.

**What changed (audit fix):**
- The conformal interval is now calibrated on a **separate calibration split** and its coverage is
  measured on a **truly held-out test set** — the earlier 0.799 was computed on the same residuals
  used to set the interval width, which is tautological. This number is honest even if it isn't 0.80.
- Dropped `year` (perfectly collinear with `age`, corr −1.0) to stabilise the SHAP explanation.

**Design:** XGBoost quantile regression (q10/q50/q90) on `log_price`; 5-fold CV for point error;
3-way 60/20/20 train/calibration/test split for the conformal interval.

In [1]:
import json, joblib
import numpy as np, pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold, train_test_split

df = pd.read_parquet("../data/processed/listings_clean.parquet")
print("rows:", len(df))

CAT = ["make","model","bodyType","transmissionType","fuelType","regionalSpecs","sellerType","city"]
NUM = ["age","kilometers","mileage_per_year","noOfCylinders"]   # dropped 'year' (collinear with age)
FEATURES = CAT + NUM
TARGET = "log_price"

cat_maps = {}
X = df[FEATURES].copy()
for c in CAT:
    cats = sorted(df[c].astype(str).unique())
    cat_maps[c] = {v:i for i,v in enumerate(cats)}
    X[c] = df[c].astype(str).map(cat_maps[c]).astype("int32")
X[NUM] = X[NUM].astype("float32")
y = df[TARGET].values
price = df["price"].values
print("features:", FEATURES)

rows: 672
features: ['make', 'model', 'bodyType', 'transmissionType', 'fuelType', 'regionalSpecs', 'sellerType', 'city', 'age', 'kilometers', 'mileage_per_year', 'noOfCylinders']


## 5-fold CV — held-out point error

In [2]:
PARAMS = dict(n_estimators=400, learning_rate=0.05, max_depth=5, subsample=0.85,
              colsample_bytree=0.85, min_child_weight=5, reg_lambda=2.0,
              random_state=42, n_jobs=-1, objective="reg:quantileerror")
Q = {"q10":0.10, "q50":0.50, "q90":0.90}
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rows=[]
for tr,te in kf.split(X):
    m = xgb.XGBRegressor(**{**PARAMS,"quantile_alpha":0.5}); m.fit(X.iloc[tr], y[tr], verbose=False)
    mid = np.expm1(m.predict(X.iloc[te])); true = price[te]
    rows.append({"MAE_AED":float(np.mean(np.abs(mid-true))),
                 "RMSE_AED":float(np.sqrt(np.mean((mid-true)**2))),
                 "MAPE_pct":float(np.mean(np.abs(mid-true)/true)*100),
                 "median_APE_pct":float(np.median(np.abs(mid-true)/true)*100)})
cv = pd.DataFrame(rows)
point = {k:{"mean":round(float(cv[k].mean()),2),"std":round(float(cv[k].std()),2)} for k in cv.columns}
print(json.dumps(point, indent=2))

{
  "MAE_AED": {
    "mean": 39237.89,
    "std": 8865.66
  },
  "RMSE_AED": {
    "mean": 114883.04,
    "std": 48702.16
  },
  "MAPE_pct": {
    "mean": 27.58,
    "std": 1.23
  },
  "median_APE_pct": {
    "mean": 19.57,
    "std": 1.49
  }
}


## Honest split-conformal interval
60% train → fit models · 20% calibration → set the interval width `delta` · 20% test → measure
coverage on data used for **neither** fitting nor calibration. This is the number we report.

In [3]:
# Repeat the 60/20/20 split over 5 seeds and average — a stable, leakage-free estimate
# (each split's test set is used for neither fitting nor calibration).
idx = np.arange(len(X)); deltas=[]; covs=[]
for seed in range(5):
    tr_i, tmp_i = train_test_split(idx, test_size=0.40, random_state=seed)
    cal_i, te_i = train_test_split(tmp_i, test_size=0.50, random_state=seed)
    m = xgb.XGBRegressor(**{**PARAMS,"quantile_alpha":0.5}); m.fit(X.iloc[tr_i], y[tr_i], verbose=False)
    d = float(np.quantile(np.abs(y[cal_i] - m.predict(X.iloc[cal_i])), 0.80))
    pred = m.predict(X.iloc[te_i])
    lo = np.expm1(pred - d); hi = np.expm1(pred + d)
    covs.append(float(np.mean((price[te_i] >= lo) & (price[te_i] <= hi)))); deltas.append(d)
conformal_delta = float(np.mean(deltas))
coverage_test = float(np.mean(covs))
print(f"per-seed coverage: {[round(c,3) for c in covs]}")
print(f"conformal_delta (log): {conformal_delta:.4f}  → band ×{np.exp(conformal_delta):.2f}")
print(f"HONEST held-out test coverage (avg of 5 splits): {coverage_test:.3f}  (target 0.80)")

per-seed coverage: [0.733, 0.696, 0.8, 0.844, 0.807]
conformal_delta (log): 0.4424  → band ×1.56
HONEST held-out test coverage (avg of 5 splits): 0.776  (target 0.80)


### Naive baseline it must beat

In [4]:
base=[]
for tr,te in kf.split(X):
    a,b = df.iloc[tr], df.iloc[te]
    mm=a.groupby(["make","model"])["price"].median(); mk=a.groupby("make")["price"].median(); g=a["price"].median()
    p=b.apply(lambda r: mm.get((r["make"],r["model"]), mk.get(r["make"], g)), axis=1).values
    base.append(float(np.mean(np.abs(p-b["price"].values))))
baseline=float(np.mean(base)); model_mae=point["MAE_AED"]["mean"]
print(f"baseline MAE {baseline:,.0f} | model MAE {model_mae:,.0f} | improvement {100*(1-model_mae/baseline):.1f}%")

baseline MAE 55,483 | model MAE 39,238 | improvement 29.3%


## Fit final models on all data, export bundle

In [5]:
final={}
for name,q in Q.items():
    m=xgb.XGBRegressor(**{**PARAMS,"quantile_alpha":q}); m.fit(X,y,verbose=False); final[name]=m
bundle={"models":final,"cat_maps":cat_maps,"features":FEATURES,"categorical_features":CAT,
        "numeric_features":NUM,"target":"log_price (expm1 -> AED)","quantiles":Q,
        "cv_metrics":point,"baseline_mae_aed":round(baseline,2),"training_rows":int(len(df)),
        "dataset":"Dubizzle UAE scrape July 2026 (672 real listings)","reference_year":2026,
        "conformal_delta_log":conformal_delta,"conformal_coverage":round(coverage_test,3),
        "conformal_method":"split-conformal: 60/20/20 train/calibration/test, coverage on held-out test"}
joblib.dump(bundle,"../backend-api/models/valuation_model.joblib",compress=3)
metrics={"model":"XGBoost quantile regression (q10/q50/q90) on log_price",
         "cv":"5-fold shuffled, seed 42 — held-out folds","metrics":point,
         "baseline_make_model_median_MAE_AED":round(baseline,2),
         "improvement_over_baseline_pct":round(100*(1-model_mae/baseline),1),
         "conformal":{"method":"split-conformal (separate calibration + held-out test)",
                      "delta_log":round(conformal_delta,4),"honest_test_coverage":round(coverage_test,3),
                      "target":0.80},
         "training_rows":int(len(df)),"features":FEATURES}
json.dump(metrics,open("../eval/valuation_metrics.json","w"),indent=2)
print(json.dumps(metrics["conformal"],indent=2))

{
  "method": "split-conformal (separate calibration + held-out test)",
  "delta_log": 0.4424,
  "honest_test_coverage": 0.776,
  "target": 0.8
}


## Honest reading
The conformal coverage is now measured on data the model never saw during fitting *or* calibration,
so it's a real out-of-sample guarantee — not circular. On 672 rows the interval is wide (that's the
data, not a bug), which is exactly why the product discloses the range and recommends inspection.